In [ ]:
import mysql.connector
from faker import Faker
import random
from datetime import timedelta

fake = Faker()

# ==========================================
# 1. DATABASE CONNECTION
# ==========================================
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="your_Sql_Passowrd",  # Update with your MySQL password
    database="DB_name"     # Updated to your new database name
)
cursor = conn.cursor()
print("Connected to hos_ms database successfully!")

# ==========================================
# 2. SAFE RESET (Prevents Duplicate Errors)
# ==========================================
print("Wiping old data to prevent duplicate errors...")
cursor.execute("SET FOREIGN_KEY_CHECKS = 0;")
tables = [
    "Biomedical", "Inter_Department_Transit", "Stock", "Mortuary", "Hygiene",
    "Blood_Transfusion", "Blood_Inventory", "CSSD", "Bill_Description", "Bill_Desk",
    "Lab_Test", "Test", "Technicians", "Lab", "Sub_dept", "OT_Technician", "OT",
    "Admissions", "Reception", "Beds", "Rooms", "Nurses", "Doctors_Schedule",
    "Doctors", "Attendance", "Equipments", "Staff", "Departments", "Patients"
]
for table in tables:
    cursor.execute(f"TRUNCATE TABLE {table};")
cursor.execute("SET FOREIGN_KEY_CHECKS = 1;")
print("Database is perfectly clean. Starting data generation...")

# ==========================================
# 3. CORE INFRASTRUCTURE (Depts, Labs, Tests)
# ==========================================
print("Generating Departments and Labs...")
departments = [
    (1,"Imaging","Block A",1,"Diagnostic"), (2,"Clinical Lab","Block B",2,"Diagnostic"),
    (3,"Cardiology","Block C",2,"Clinical"), (4,"Neurology","Block C",3,"Clinical"),
    (5,"Orthopedics","Block D",1,"Clinical"), (6,"Emergency","Block A",0,"Clinical"),
    (7,"Pediatrics","Block E",1,"Clinical"), (8,"Gynecology","Block E",2,"Clinical"),
    (9,"ENT","Block D",2,"Clinical"), (10,"Biomedical","Block F",-1,"Support"),
    (11,"Mortuary","Block F",-1,"Support"), (12,'Blood Bank','Block C',1,'Support')
]
for d in departments:
    cursor.execute("INSERT INTO Departments (Department_Id, Department, Block, Floor, Dept_Type) VALUES (%s,%s,%s,%s,%s)", d)

sub = [
    (1,'X-Ray',1,1), (2,'CT Scan',1,1), (3,'MRI',1,1), (4,'Ultrasound',1,1), 
    (5,'Pathology',2,1), (6,'Biochemistry',2,1), (7,'Microbiology',2,1)
]
for s in sub:
    cursor.execute("INSERT INTO Sub_dept (Sub_Dept_ID, Sub_Dept_name, Dept_Id, Floor) VALUES (%s,%s,%s,%s)", s)

labs = [(1, 'Main Path Lab', 5), (2, 'Stat Lab', 6), (3, 'Micro Wing', 7)]
for lab in labs:
    cursor.execute("INSERT INTO Lab (Lab_id, Lab_Name, Sub_Dept_ID) VALUES (%s, %s, %s)", lab)

tests = [(1, "Blood Sugar", 70, 110), (2, "Hemoglobin", 12, 17), (3, "WBC Count", 4000, 11000)]
for t in tests:
    cursor.execute("INSERT INTO Test VALUES(%s,%s,500,%s,%s,'30m')", t)

# ==========================================
# 4. STAFF & ROLES (1050 Total)
# ==========================================
print("Generating Staff members...")
staff_names = {}

for i in range(1, 1051):
    if 1 <= i <= 250: role = "Doctor"
    elif 251 <= i <= 750: role = "Nurse"
    elif 751 <= i <= 900: role = "Hygiene"
    elif 901 <= i <= 1000: role = "Technician"
    else: role = "Reception"

    name = fake.name()[:30] # Limit to 30 chars
    staff_names[i] = name

    cursor.execute("""
    INSERT INTO Staff VALUES(%s,%s,%s,%s,%s,%s,%s,%s)
    """, (i, name, fake.date_of_birth(minimum_age=26, maximum_age=60), fake.date_between("-10y", "today"),
          fake.phone_number()[:10], role, random.choice(["M", "F"]), fake.phone_number()[:10]))

print("Assigning Staff Roles & Schedules...")
for i in range(1, 251): # Doctors
    cursor.execute("INSERT INTO Doctors VALUES(%s,%s,%s,%s)", (i, staff_names[i], "General", random.randint(1,6)))
    cursor.execute("INSERT INTO Doctors_Schedule VALUES (%s,%s,%s,%s,%s)", (i, i, "09:00 AM - 05:00 PM", "Mon-Fri", random.randint(500, 2000)))

for i in range(251, 751): # Nurses
    cursor.execute("INSERT INTO Nurses VALUES(%s,%s)", (i, random.randint(1,6)))

for i in range(751, 901): # Hygiene
    cursor.execute("INSERT INTO Hygiene VALUES (%s,%s,%s,%s)", (i, random.randint(1,12), "Morning", random.choice([True, False])))

for i in range(901, 1001): # Technicians
    cursor.execute("INSERT INTO Technicians VALUES (%s,%s,%s)", (i, random.randint(1,3), "Lab Tech"))

for i in range(1001, 1051): # Reception
    cursor.execute("INSERT INTO Reception VALUES (%s,%s)", (i, "Block A"))

# ==========================================
# 5. EQUIPMENTS, STOCK & TRANSIT
# ==========================================
print("Generating Equipments and Inventory...")
eq_catalog = [("MRI Scanner", "Diagnostic"), ("CT Scanner", "Diagnostic"), ("Ventilator", "Life Support"), ("Autoclave", "Sterilization")]
for i in range(1, 201):
    eq_name, eq_class = random.choice(eq_catalog)
    cursor.execute("INSERT INTO Equipments VALUES (%s,%s,%s,%s,%s,%s)", (i, eq_name, eq_class, "BrandX", f"Mod-{i}", random.randint(1, 12)))
    # Add to Stock table
    cursor.execute("INSERT INTO Stock VALUES (%s, %s)", (i, i))

# Inter-Department Transit (50 movements)
for i in range(1, 51):
    cursor.execute("INSERT INTO Inter_Department_Transit VALUES (%s, NOW(), %s, %s, %s)", 
                   (i, random.randint(1,12), random.randint(1,12), random.randint(1,200)))

# ==========================================
# 6. PATIENTS, ROOMS, ADMISSIONS
# ==========================================
print("Generating Patients and Admissions...")
for i in range(1, 1001):
    cursor.execute("""
    INSERT INTO Patients (Patient_Name, DoB, Gender, Phone_Number, Guardian, Alt_Number, Aadhaar)
    VALUES (%s,%s,%s,%s,%s,%s,%s)
    """, (fake.name()[:20], fake.date_of_birth(minimum_age=1, maximum_age=90), random.choice(["M","F"]),
          fake.phone_number()[:10], fake.name()[:20], fake.phone_number()[:10], str(random.randint(100000000000, 999999999999))))

for i in range(1, 201):
    cursor.execute("INSERT INTO Rooms (Room_Number, Room_Type, Department_Id) VALUES(%s,%s,%s)", (str(100 + i), "General Ward", random.randint(1, 6)))

bed_id = 1
for room in range(1, 201):
    for b in range(2):
        cursor.execute("INSERT INTO Beds (Room_Id, Bed_Identifier, Daily_Charge) VALUES(%s,%s,%s)", (room, chr(65 + b), random.randint(1000, 5000)))
        bed_id += 1

for i in range(1, 1001):
    cursor.execute("INSERT INTO Admissions (Patient_Id, Admission_Date, Is_Discharged, Doctor_assigned, Bed_No) VALUES (%s, NOW() - INTERVAL %s DAY, %s, %s, %s)", 
                   (random.randint(1, 1000), random.randint(0, 30), random.choice([0, 1]), random.randint(1, 250), random.randint(1, 400)))

# ==========================================
# 7. HOSPITAL OPERATIONS (Bills, Blood, OT, etc.)
# ==========================================
print("Generating Bills, Lab Tests, and Final Operations...")

# Lab Tests (1000)
for i in range(1, 1001):
    cursor.execute("INSERT INTO Lab_Test VALUES(%s,%s,%s,NOW(),%s,%s,NOW() + INTERVAL 2 HOUR)", 
                   (i, random.randint(1, 1000), random.randint(1, 3), random.randint(5, 500), random.choice([0, 1])))

# Bills (Using Receptionists for Billing)
desc_id = 1
for bill_no in range(1, 501):
    amt = random.randint(2000, 15000)
    cursor.execute("INSERT INTO Bill_Desk VALUES (%s,%s,%s,NOW(),%s,%s)", (bill_no, random.randint(1001, 1050), random.randint(1, 1000), amt, amt))
    cursor.execute("INSERT INTO Bill_Description VALUES (%s,%s,%s,%s)", (desc_id, bill_no, "Room Charges", amt))
    desc_id += 1

# Blood Bank
for i in range(1, 201):
    cursor.execute("INSERT INTO Blood_Inventory (Blood_Group, Component_Type, Collection_Date, Expiry_Date, Dept_Id) VALUES (%s,%s,NOW(),NOW() + INTERVAL 40 DAY,%s)", 
                   (random.choice(['A+', 'O+', 'B+']), 'Whole Blood', 12))
for i in range(1, 51):
    cursor.execute("INSERT INTO Blood_Transfusion (Blood_Unit_Id, Patient_Id, Doctor_Id, Transfusion_Date) VALUES (%s,%s,%s,NOW())", 
                   (random.randint(1, 200), random.randint(1, 1000), random.randint(1, 250)))

# Operations & Mortuary
for i in range(1, 11):
    cursor.execute("INSERT INTO OT VALUES (%s,%s,%s,%s,NOW(),NOW() + INTERVAL 2 HOUR)", (i, random.randint(1,9), "Block A", True))
    cursor.execute("INSERT INTO OT_Technician VALUES (%s,%s)", (i, random.randint(1,9)))
    cursor.execute("INSERT INTO Mortuary VALUES (%s, NOW(), %s, %s, %s)", (random.randint(1, 1000), "Cardiac Arrest", 11, random.randint(1, 9)))

# CSSD & Biomedical
for i in range(1, 51):
    cursor.execute("INSERT INTO CSSD VALUES (%s, NOW(), %s, %s, %s, %s)", (random.randint(1, 200), random.randint(251, 750), "Autoclave", "Critical", random.randint(1, 12)))
    cursor.execute("INSERT INTO Biomedical (Category, Equip_Id, Dept_Id, Down_Time, Resolved_Time, Repair_Cost, Company_Incharge, Company_No) VALUES (%s,%s,%s,NOW() - INTERVAL 5 DAY, NOW(), %s, %s, %s)", 
                   ("Maintenance", random.randint(1, 200), random.randint(1, 12), random.randint(1000, 5000), "FixIt Co", "1234567890"))

# Attendance (1000 records)
for i in range(1, 1001):
    cursor.execute("INSERT INTO Attendance VALUES (%s,%s,%s,NOW(),'08:00:00','17:00:00','Present')", (i, random.randint(1, 1050), random.randint(1,12)))

# ==========================================
# 8. COMMIT AND CLOSE
# ==========================================
conn.commit()
cursor.close()
conn.close()
print("=========================================")
print("SUCCESS! Entire hospital database populated.")
print("=========================================")

Connected to hos_ms database successfully!
Wiping old data to prevent duplicate errors...
Database is perfectly clean. Starting data generation...
Generating Departments and Labs...
Generating Staff members...
Assigning Staff Roles & Schedules...
Generating Equipments and Inventory...
Generating Patients and Admissions...
Generating Bills, Lab Tests, and Final Operations...
SUCCESS! Entire hospital database populated.
